# Dense-set modelling — traceable, model by model

A flat rewrite of `02_dense_modelling.ipynb` with **no project-defined functions**.
Everything shared is a *constant* or a *precomputed data structure*, so each model
can be read top to bottom, and every loop prints what it just did.

| block | what it does | touches |
|---|---|---|
| 0 | constants | — |
| 1 | folds, built once as data | `train` |
| 2a–2d | one block per model, CV + final refit | `train` |
| 3 | CV metrics and model comparison | — |
| 4 | freeze the selected model | — |
| 5 | **sealed FE test — the only place `test` is predicted** | `test` |
| 6 | FE metrics, pass criteria, decade coverage | — |

Set `VERBOSE = 0` at the top to silence the tracing once it's understood.

## Block 0 — constants

No functions. Every knob in one place.

In [94]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1]))
from src.data import load_data
from IPython.display import display
from scipy.spatial.distance import pdist
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.compose import TransformedTargetRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel

RANDOM_STATE = 42

# target -> physical floor. mu_r cannot go below 1, rho cannot go below 0.
TARGETS = {'permeability': 1.0, 'resistivity': 0.0}

# Target transform, chosen per target rather than applied uniformly.
#   mu_r spans 1000x (ln range 6.91 nats), so absolute-error fitting is dominated by
#   the high end and the low end runs wild. It MUST be fitted in log space.
#   rho spans only 5x (ln range 1.61 nats), where the log and linear scales are very
#   nearly affine-equivalent, so the transform buys almost no reweighting -- and
#   one-level-out CV prefers the raw target. Fitted raw; the physical floor is
#   enforced at reporting time via TARGETS rather than structurally by exp().
# Both arms go through TransformedTargetRegressor so that `.regressor_` is uniform;
# (None, None) is sklearn's identity transform.
TARGET_FUNC = {'permeability': (np.log, np.exp), 'resistivity': (None, None)}

# Bounds used in reporting. NOTHING IS EVER CLIPPED anywhere in this notebook -- these
# only COUNT violations, on the raw predictions, so that a model leaving the physical or
# the supported region stays visible instead of being silently repaired.
#
# Physical bounds: properties of the material class, independent of this dataset.
#   mu_r >= 1  -- steel is ferromagnetic through the transformation, never diamagnetic
#   rho  >  0  -- any normal conductor
# Stored as (floor, strict); strict=True means the floor value itself is a violation.
PHYS_BOUNDS = {'permeability': (1.0, False), 'resistivity': (0.0, True)}

# Training-range bounds: the support the model was actually fitted on. A prediction
# outside these is extrapolation -- physically possible but unsupported by the data,
# and data/raw/train_readme.txt explicitly says not to extrapolate beyond them.
RANGE_BOUNDS = {'permeability': (1.0, 1000.0), 'resistivity': (2e-7, 1e-6)}

# How many adjacent levels each held-out band spans.
# mu grid is LOG-uniform: a 2-NN average of the flanking levels lands exactly on a
# single held-out level (geometric mean), so a 1-level band lets kNN reconstruct the
# answer. Bands of >=2 break that. The rho grid is LINEAR, so the artifact cannot
# occur, and rho only has 7 interior levels to spend.
BAND_WIDTH = {'permeability': 2, 'resistivity': 1}

# Selection metric. TransformedTargetRegressor sits OUTSIDE GridSearchCV, so the
# search sees the TRANSFORMED target: for mu_r that is log(y), where MAE is the
# absolute log-error (~MAPE/100); for rho it is y itself, where MAE is in ohm m and
# so weights absolute rather than relative error. Over rho's 5x range the two orderings
# barely differ, which is the same reason the transform itself is not needed there.
# R^2 is unusable: a fold holding one level has zero target variance, so every
# candidate scores 0.0 and the search silently returns the first grid entry.
SCORING = 'neg_mean_absolute_error'

# FE TEST ONLY. Never applied to cross-validation tables.
PASS_OVERALL, PASS_LEVEL = 10.0, 20.0

VERBOSE     = 2   # 0 silent | 1 one line per outer fold | 2 also inner-CV tables
TRACE_FOLDS = 2   # at VERBOSE=2, full tuning table for the first N folds

FLOOR = 'kNN (PCA-4)'   # memorisation baseline every model must beat

pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
pd.set_option('display.width', 200)
print('constants set')

constants set


In [95]:
train = load_data('train.xlsx')
test  = load_data('test.xlsx')

# One list, not a function: train and test share column identity (asserted below).
H = [c for c in train.columns if c.startswith(('H_real_', 'H_imag_'))]

assert list(train.columns) == list(test.columns), 'column order differs'
assert len(H) == 16, f'expected 16 H channels, got {len(H)}'
# NB: checked on the whole frame rather than on the feature columns, so that the
# expression selecting FE features appears in exactly ONE cell in this notebook --
# the sealed test cell in Block 5. Search the notebook for it to confirm.
assert train.isna().sum().sum() == 0 and test.isna().sum().sum() == 0
assert train.duplicated(['permeability', 'resistivity']).sum() == 0
for t in TARGETS:
    assert test[t].min() >= train[t].min() and test[t].max() <= train[t].max(), \
        f'{t} extrapolates outside the training range'

print(f'train {train.shape}  {train.permeability.nunique()} mu x {train.resistivity.nunique()} rho')
print(f'test  {test.shape}   {test.permeability.nunique()} mu x {test.resistivity.nunique()} rho')
print(f'features: {H[:2]} ... {H[-2:]}')

train (225, 18)  25 mu x 9 rho
test  (55, 18)   11 mu x 5 rho
features: ['H_imag_375', 'H_imag_750'] ... ['H_real_24000', 'H_real_48000']


## Block 1 — folds, built once as data

Replaces `band_splits` / `level_folds`. Two dictionaries:

- `FOLDS[target]` = list of `(tr_idx, te_idx, inner_cv)` — the outer folds
- `FULL_INNER[target]` = inner CV over all 225 rows, for the final refit

`inner_cv` positions are relative to `train.iloc[tr_idx]`, which is what
`GridSearchCV` indexes into. Every model block below contains **zero** splitting
logic — it just consumes these.

The first and last level of each axis are anchors and are never held out, which is
what keeps the protocol interpolation-only.

In [96]:
FOLDS, FULL_INNER = {}, {}

for target, bw in BAND_WIDTH.items():
    values   = train[target].values
    levels   = np.sort(train[target].unique())
    interior = levels[1:-1]                      # anchors excluded, never held out
    n_bands  = max(1, len(interior) // bw)

    print(f'\n{target} | band_width={bw} | {len(levels)} levels '
          f'({len(interior)} interior, anchors {levels[0]:g} and {levels[-1]:g}) '
          f'-> {n_bands} outer bands')

    outer = []
    for i, band in enumerate(np.array_split(interior, n_bands)):
        te = np.where(np.isin(values, band))[0]
        tr = np.where(~np.isin(values, band))[0]

        # inner folds, positional WITHIN train.iloc[tr]
        sub_vals = values[tr]
        sub_int  = np.sort(np.unique(sub_vals))[1:-1]
        inner    = []
        for iband in np.array_split(sub_int, max(1, len(sub_int) // bw)):
            im = np.isin(sub_vals, iband)
            inner.append((np.where(~im)[0], np.where(im)[0]))

        outer.append((tr, te, inner))
        shown = '[' + ', '.join(f'{v:.4g}' for v in band) + ']'   # .4g keeps rho readable
        print(f'  fold {i:2d} | held out {shown:34s} '
              f'| train {len(tr):3d} / test {len(te):3d} | {len(inner)} inner folds')

    FOLDS[target] = outer

    # inner CV over the FULL training set, used for the final refit in each block
    full_int = levels[1:-1]
    FULL_INNER[target] = []
    for iband in np.array_split(full_int, max(1, len(full_int) // bw)):
        im = np.isin(values, iband)
        FULL_INNER[target].append((np.where(~im)[0], np.where(im)[0]))
    print(f'  full-set inner CV: {len(FULL_INNER[target])} folds over all {len(train)} rows')


permeability | band_width=2 | 25 levels (23 interior, anchors 1 and 1000) -> 11 outer bands
  fold  0 | held out [1.334, 1.778, 2.371]              | train 198 / test  27 | 10 inner folds
  fold  1 | held out [3.162, 4.217]                     | train 207 / test  18 | 10 inner folds
  fold  2 | held out [5.623, 7.499]                     | train 207 / test  18 | 10 inner folds
  fold  3 | held out [10, 13.34]                        | train 207 / test  18 | 10 inner folds
  fold  4 | held out [17.78, 23.71]                     | train 207 / test  18 | 10 inner folds
  fold  5 | held out [31.62, 42.17]                     | train 207 / test  18 | 10 inner folds
  fold  6 | held out [56.23, 74.99]                     | train 207 / test  18 | 10 inner folds
  fold  7 | held out [100, 133.4]                       | train 207 / test  18 | 10 inner folds
  fold  8 | held out [177.8, 237.1]                     | train 207 / test  18 | 10 inner folds
  fold  9 | held out [316.2, 421.7]        

In [97]:
# Protocol assertions -- run for BOTH targets.
for target, bw in BAND_WIDTH.items():
    anchors = set(np.sort(train[target].unique())[[0, -1]])
    for tr, te, inner in FOLDS[target]:
        held = set(train.iloc[te][target])
        assert len(set(tr) & set(te)) == 0,            'train/test overlap'
        assert len(tr) + len(te) == len(train),        'rows dropped'
        assert not held & set(train.iloc[tr][target]), 'held-out level leaked into training'
        assert anchors <= set(train.iloc[tr][target]), 'anchor missing from training'
        assert len(held) >= bw,   f'band narrower than band_width={bw}'
        assert len(inner) >= 2,   f'{target}: only {len(inner)} inner fold(s) - cannot select'
        for itr, ival in inner:
            assert len(set(itr) & set(ival)) == 0, 'inner overlap'
            assert itr.max() < len(tr) and ival.max() < len(tr), 'inner index out of range'
    print(f'{target:13s} OK: {len(FOLDS[target])} outer x '
          f'{len(FOLDS[target][0][2])} inner, no leakage, anchors held')

permeability  OK: 11 outer x 10 inner, no leakage, anchors held
resistivity   OK: 7 outer x 6 inner, no leakage, anchors held


## Block 1b — hyperparameter grids and kernel bounds

The GP length scale is measured in standardised H space, so it is only meaningful
between the closest pair of training rows and the diameter of the set. Below the
minimum pairwise distance the kernel separates nothing; above the diameter it is
constant everywhere. The bounds are set from that geometry, with a decade of
margin, and are asserted non-binding in the traces below — a fitted length scale
sitting *on* a bound would mean the bound is shaping the fit.

In [98]:
Z  = StandardScaler().fit_transform(train[H])
pw = pdist(Z, metric='euclidean')
D_MIN, D_MAX = pw.min(), pw.max()

ELL_BOUNDS = (D_MIN / 10, D_MAX * 10)   # one decade of margin each side
SF_BOUNDS  = (1e-3, 1e8)                # output scale lives in target space
ALPHAS     = np.logspace(-6, 6, 25)     # Poly2 ridge penalty
NUGGETS    = [1e-8, 1e-6, 1e-4, 1e-3]   # GP nugget (sigma^2 added to K's diagonal)

print(f'pairwise distance: min {D_MIN:.4f} | median {np.median(pw):.3f} | diameter {D_MAX:.3f}')
print(f'ELL_BOUNDS = ({ELL_BOUNDS[0]:.4g}, {ELL_BOUNDS[1]:.4g})')
print(f'SF_BOUNDS  = {SF_BOUNDS}')
print(f'ALPHAS     = {ALPHAS.size} values, {ALPHAS[0]:.0e} .. {ALPHAS[-1]:.0e}')
print(f'NUGGETS    = {NUGGETS}')

pairwise distance: min 0.0781 | median 4.411 | diameter 13.373
ELL_BOUNDS = (0.007807, 133.7)
SF_BOUNDS  = (0.001, 100000000.0)
ALPHAS     = 25 values, 1e-06 .. 1e+06
NUGGETS    = [1e-08, 1e-06, 0.0001, 0.001]


## Blocks 2a–2d — one block per model

Each block is self-contained: it declares its pipeline literally, loops over
`FOLDS[target]`, and appends predictions to the shared `cv_rows` list. Nothing is
shared between blocks except the folds and the constants.

At `VERBOSE=2` the first fold of each (model, target) prints the **full inner-CV
table** — every candidate, its score on each inner fold, its mean and rank — so the
hyperparameter selection is visible rather than asserted.

In [99]:
cv_rows     = []   # every model block appends prediction frames here
FINAL       = {}   # (target, model) -> model refit on all 225 training rows
kernel_rows = []   # GP blocks: one row PER FOLD -- fitted ell, sf, LML, convergence
print('cv_rows, FINAL and kernel_rows initialised')

cv_rows, FINAL and kernel_rows initialised


### 2a — kNN (PCA-4): the memorisation floor

Pure lookup after whitening to 4 components. Every other model must beat it; if one doesn't, that model has learned nothing the grid didn't already contain.

In [100]:
NAME = 'kNN (PCA-4)'
PIPE = Pipeline([('scaler', StandardScaler()),
                 ('pca',    PCA(n_components=4, whiten=True)),
                 ('model',  KNeighborsRegressor())])
GRID = {'model__n_neighbors': [1, 2, 3, 5, 8],
        'model__weights': ['uniform', 'distance']}

print(f'=== {NAME} ===')
print(f'grid: {GRID}')

for target in TARGETS:
    fold_mapes, fold_picks = [], []
    for i, (tr, te, inner) in enumerate(FOLDS[target]):
        mdl = TransformedTargetRegressor(
            GridSearchCV(clone(PIPE), GRID, cv=inner, scoring=SCORING, n_jobs=-1),
            func=TARGET_FUNC[target][0], inverse_func=TARGET_FUNC[target][1])
        mdl.fit(train[H].iloc[tr], train[target].iloc[tr])

        pred = mdl.predict(train[H].iloc[te])
        yt   = train[target].iloc[te].values
        fold_mape = np.mean(np.abs((pred - yt) / yt)) * 100
        fold_mapes.append(fold_mape)
        fold_picks.append(str(mdl.regressor_.best_params_))

        if VERBOSE >= 1:
            shown = '[' + ', '.join(f'{v:.4g}' for v in np.unique(yt)) + ']'
            print(f'  {target:12s} fold {i:2d} | held out {shown:26s} '
                  f'| fit on {train[H].iloc[tr].shape} | picked {mdl.regressor_.best_params_} '
                  f'| fold MAPE {fold_mape:8.3f}%')
        # ---- what the hyperparameter search actually did ----
        gs  = mdl.regressor_
        if VERBOSE >= 2 and i < TRACE_FOLDS:
            res = gs.cv_results_
            tune = pd.DataFrame({'candidate': [str(p) for p in res['params']]})
            for j in range(gs.n_splits_):
                tune[f'f{j}'] = res[f'split{j}_test_score']
            tune['mean'] = res['mean_test_score']
            tune['rank'] = res['rank_test_score']
            print(f'    inner CV: {gs.n_splits_} folds, scoring={SCORING} (higher is better)')
            print('      ' + tune.round(5).to_string(index=False).replace('\n', '\n      '))
            print(f'    -> selected {gs.best_params_} (mean {gs.best_score_:.5f})')

        rows = train.iloc[te]
        cv_rows.append(pd.DataFrame({'target': target, 'model': NAME,
                                     'permeability': rows.permeability.values,
                                     'resistivity':  rows.resistivity.values,
                                     'y_true': yt, 'y_pred': pred,
                                     'picked': str(mdl.regressor_.best_params_)}))

    print(f'  -> {target}: {len(fold_mapes)} folds, median fold MAPE {np.median(fold_mapes):.3f}%')

    # final refit on ALL 225 TRAINING rows. No test data is involved.
    mdl_full = TransformedTargetRegressor(
        GridSearchCV(clone(PIPE), GRID, cv=FULL_INNER[target], scoring=SCORING, n_jobs=-1),
        func=TARGET_FUNC[target][0], inverse_func=TARGET_FUNC[target][1])
    mdl_full.fit(train[H], train[target])
    FINAL[(target, NAME)] = mdl_full

    # Does the full-set refit agree with what the folds chose? If not, the CV
    # estimate describes a different model from the one that will be tested.
    chose = pd.Series(fold_picks).value_counts()
    modal = set(chose[chose == chose.max()].index)      # may be a tie
    print(f'  -> final refit on {train[H].shape}: {mdl_full.regressor_.best_params_}')
    print(f'     {len(fold_picks)} folds chose:')
    for k, v in chose.items():
        print(f'        {v:2d}x  {k}')
    print(f'     full-set pick is among the modal fold choices: '
          f'{str(mdl_full.regressor_.best_params_) in modal}'
          + ('   (folds tied)' if len(modal) > 1 else ''))

=== kNN (PCA-4) ===
grid: {'model__n_neighbors': [1, 2, 3, 5, 8], 'model__weights': ['uniform', 'distance']}
  permeability fold  0 | held out [1.334, 1.778, 2.371]      | fit on (198, 16) | picked {'model__n_neighbors': 3, 'model__weights': 'uniform'} | fold MAPE   39.930%
    inner CV: 10 folds, scoring=neg_mean_absolute_error (higher is better)
                                                    candidate     f0     f1     f2     f3     f4     f5     f6     f7     f8     f9   mean  rank
       {'model__n_neighbors': 1, 'model__weights': 'uniform'} -0.432 -0.288 -0.288 -0.288 -0.288 -0.432 -0.384 -0.464 -0.512 -0.432 -0.381     9
      {'model__n_neighbors': 1, 'model__weights': 'distance'} -0.432 -0.288 -0.288 -0.288 -0.288 -0.432 -0.384 -0.464 -0.512 -0.432 -0.381     9
       {'model__n_neighbors': 2, 'model__weights': 'uniform'} -0.408 -0.288 -0.288 -0.304 -0.288 -0.328 -0.344 -0.416 -0.456 -0.376 -0.349     8
      {'model__n_neighbors': 2, 'model__weights': 'distance'} -0.338 -

### 2b — Poly2 + Ridge

Degree-2 polynomial over the 16 channels (152 terms), scaled AFTER expansion so the terms are comparable, with the ridge penalty chosen by inner CV.

In [101]:
NAME = 'Poly2+Ridge'
PIPE = Pipeline([('poly',   PolynomialFeatures(degree=2, include_bias=False)),
                 ('scaler', StandardScaler()),
                 ('model',  Ridge())])
GRID = {'model__alpha': ALPHAS}

print(f'=== {NAME} ===')
print(f'grid: {GRID}')
print(f'degree-2 expansion: {PIPE.named_steps["poly"].fit(train[H]).n_output_features_} terms')

for target in TARGETS:
    fold_mapes, fold_picks = [], []
    for i, (tr, te, inner) in enumerate(FOLDS[target]):
        mdl = TransformedTargetRegressor(
            GridSearchCV(clone(PIPE), GRID, cv=inner, scoring=SCORING, n_jobs=-1),
            func=TARGET_FUNC[target][0], inverse_func=TARGET_FUNC[target][1])
        mdl.fit(train[H].iloc[tr], train[target].iloc[tr])

        pred = mdl.predict(train[H].iloc[te])
        yt   = train[target].iloc[te].values
        fold_mape = np.mean(np.abs((pred - yt) / yt)) * 100
        fold_mapes.append(fold_mape)
        fold_picks.append(str(mdl.regressor_.best_params_))

        if VERBOSE >= 1:
            shown = '[' + ', '.join(f'{v:.4g}' for v in np.unique(yt)) + ']'
            print(f'  {target:12s} fold {i:2d} | held out {shown:26s} '
                  f'| fit on {train[H].iloc[tr].shape} | picked {mdl.regressor_.best_params_} '
                  f'| fold MAPE {fold_mape:8.3f}%')
        # ---- what the hyperparameter search actually did ----
        gs  = mdl.regressor_
        if VERBOSE >= 2 and i < TRACE_FOLDS:
            res = gs.cv_results_
            tune = pd.DataFrame({'candidate': [str(p) for p in res['params']]})
            for j in range(gs.n_splits_):
                tune[f'f{j}'] = res[f'split{j}_test_score']
            tune['mean'] = res['mean_test_score']
            tune['rank'] = res['rank_test_score']
            print(f'    inner CV: {gs.n_splits_} folds, scoring={SCORING} (higher is better)')
            print('      ' + tune.round(5).to_string(index=False).replace('\n', '\n      '))
            print(f'    -> selected {gs.best_params_} (mean {gs.best_score_:.5f})')

        rows = train.iloc[te]
        cv_rows.append(pd.DataFrame({'target': target, 'model': NAME,
                                     'permeability': rows.permeability.values,
                                     'resistivity':  rows.resistivity.values,
                                     'y_true': yt, 'y_pred': pred,
                                     'picked': str(mdl.regressor_.best_params_)}))

    print(f'  -> {target}: {len(fold_mapes)} folds, median fold MAPE {np.median(fold_mapes):.3f}%')

    # final refit on ALL 225 TRAINING rows. No test data is involved.
    mdl_full = TransformedTargetRegressor(
        GridSearchCV(clone(PIPE), GRID, cv=FULL_INNER[target], scoring=SCORING, n_jobs=-1),
        func=TARGET_FUNC[target][0], inverse_func=TARGET_FUNC[target][1])
    mdl_full.fit(train[H], train[target])
    FINAL[(target, NAME)] = mdl_full

    # Does the full-set refit agree with what the folds chose? If not, the CV
    # estimate describes a different model from the one that will be tested.
    chose = pd.Series(fold_picks).value_counts()
    modal = set(chose[chose == chose.max()].index)      # may be a tie
    print(f'  -> final refit on {train[H].shape}: {mdl_full.regressor_.best_params_}')
    print(f'     {len(fold_picks)} folds chose:')
    for k, v in chose.items():
        print(f'        {v:2d}x  {k}')
    print(f'     full-set pick is among the modal fold choices: '
          f'{str(mdl_full.regressor_.best_params_) in modal}'
          + ('   (folds tied)' if len(modal) > 1 else ''))

=== Poly2+Ridge ===
grid: {'model__alpha': array([1.00000000e-06, 3.16227766e-06, 1.00000000e-05, 3.16227766e-05,
       1.00000000e-04, 3.16227766e-04, 1.00000000e-03, 3.16227766e-03,
       1.00000000e-02, 3.16227766e-02, 1.00000000e-01, 3.16227766e-01,
       1.00000000e+00, 3.16227766e+00, 1.00000000e+01, 3.16227766e+01,
       1.00000000e+02, 3.16227766e+02, 1.00000000e+03, 3.16227766e+03,
       1.00000000e+04, 3.16227766e+04, 1.00000000e+05, 3.16227766e+05,
       1.00000000e+06])}
degree-2 expansion: 152 terms
  permeability fold  0 | held out [1.334, 1.778, 2.371]      | fit on (198, 16) | picked {'model__alpha': np.float64(3.1622776601683795e-05)} | fold MAPE    5.421%
    inner CV: 10 folds, scoring=neg_mean_absolute_error (higher is better)
                                                 candidate     f0     f1     f2     f3     f4     f5     f6     f7     f8     f9   mean  rank
                       {'model__alpha': np.float64(1e-06)} -0.055 -0.047 -0.037 -0.056 -0.096 -

### 2c / 2d — Gaussian Process

Two tuning mechanisms run at once, which is the confusing part:

- the **nugget** (`alpha`) is chosen by inner CV, like every other model
- the **length scale** and **output scale** are fitted by *marginal likelihood*
  inside each `.fit()`, and never appear in `GridSearchCV`

At `VERBOSE=2` both are printed for the traced fold. `ell/diameter > 1` means the
optimiser has stretched the kernel past the whole dataset to compensate for
assuming more roughness than the data has — a misspecification signature.

In [102]:
NAME = 'GPR (Matern 1.5)'
PIPE = Pipeline([('scaler', StandardScaler()),
                 ('model',  GaussianProcessRegressor(
                     kernel=ConstantKernel(1.0, SF_BOUNDS) * Matern(1.0, ELL_BOUNDS, nu=1.5),
                     normalize_y=True,        # fallback is the training mean, not 0
                     n_restarts_optimizer=0,  # 2 free hyperparameters, unimodal surface:
                                              # 0/1/2/5/10 verified identical
                     random_state=RANDOM_STATE))])
GRID = {'model__alpha': NUGGETS}

print(f'=== {NAME} ===')
print(f'grid: {GRID}')

for target in TARGETS:
    fold_mapes, fold_picks = [], []
    for i, (tr, te, inner) in enumerate(FOLDS[target]):
        mdl = TransformedTargetRegressor(
            GridSearchCV(clone(PIPE), GRID, cv=inner, scoring=SCORING, n_jobs=-1),
            func=TARGET_FUNC[target][0], inverse_func=TARGET_FUNC[target][1])
        mdl.fit(train[H].iloc[tr], train[target].iloc[tr])

        pred = mdl.predict(train[H].iloc[te])
        yt   = train[target].iloc[te].values
        fold_mape = np.mean(np.abs((pred - yt) / yt)) * 100
        fold_mapes.append(fold_mape)
        fold_picks.append(str(mdl.regressor_.best_params_))

        if VERBOSE >= 1:
            shown = '[' + ', '.join(f'{v:.4g}' for v in np.unique(yt)) + ']'
            print(f'  {target:12s} fold {i:2d} | held out {shown:26s} '
                  f'| fit on {train[H].iloc[tr].shape} | picked {mdl.regressor_.best_params_} '
                  f'| fold MAPE {fold_mape:8.3f}%')
        # ---- what the hyperparameter search actually did ----
        gs  = mdl.regressor_
        # Kernel diagnostics for EVERY fold, not just the traced ones, so that the
        # length-scale evidence in Block 4 is read off a table instead of recalled.
        # ell and sf are fitted by marginal likelihood inside .fit(); alpha comes from
        # the inner CV. Recorded together because Block 4 compares them at matched alpha.
        _g = gs.best_estimator_.named_steps['model']
        kernel_rows.append({'kernel': NAME, 'target': target, 'fold': i,
                            'alpha': gs.best_params_['model__alpha'],
                            'ell':   _g.kernel_.k2.length_scale,
                            'sf':    np.sqrt(_g.kernel_.k1.constant_value),
                            'lml':   _g.log_marginal_likelihood_value_,
                            'ell_over_diam': _g.kernel_.k2.length_scale / D_MAX})
        if VERBOSE >= 2 and i < TRACE_FOLDS:
            res = gs.cv_results_
            tune = pd.DataFrame({'candidate': [str(p) for p in res['params']]})
            for j in range(gs.n_splits_):
                tune[f'f{j}'] = res[f'split{j}_test_score']
            tune['mean'] = res['mean_test_score']
            tune['rank'] = res['rank_test_score']
            print(f'    inner CV: {gs.n_splits_} folds, scoring={SCORING} (higher is better)')
            print('      ' + tune.round(5).to_string(index=False).replace('\n', '\n      '))
            print(f'    -> selected {gs.best_params_} (mean {gs.best_score_:.5f})')
            g = _g
            print(f'    MLE-fitted kernel (the OTHER half of the tuning): '
                  f'ell={g.kernel_.k2.length_scale:.3f}  '
                  f'sf={np.sqrt(g.kernel_.k1.constant_value):.3f}  '
                  f'LML={g.log_marginal_likelihood_value_:.2f}  '
                  f'ell/diameter={g.kernel_.k2.length_scale / D_MAX:.2f}')

        rows = train.iloc[te]
        cv_rows.append(pd.DataFrame({'target': target, 'model': NAME,
                                     'permeability': rows.permeability.values,
                                     'resistivity':  rows.resistivity.values,
                                     'y_true': yt, 'y_pred': pred,
                                     'picked': str(mdl.regressor_.best_params_)}))

    print(f'  -> {target}: {len(fold_mapes)} folds, median fold MAPE {np.median(fold_mapes):.3f}%')

    # final refit on ALL 225 TRAINING rows. No test data is involved.
    mdl_full = TransformedTargetRegressor(
        GridSearchCV(clone(PIPE), GRID, cv=FULL_INNER[target], scoring=SCORING, n_jobs=-1),
        func=TARGET_FUNC[target][0], inverse_func=TARGET_FUNC[target][1])
    mdl_full.fit(train[H], train[target])
    FINAL[(target, NAME)] = mdl_full

    # Does the full-set refit agree with what the folds chose? If not, the CV
    # estimate describes a different model from the one that will be tested.
    chose = pd.Series(fold_picks).value_counts()
    modal = set(chose[chose == chose.max()].index)      # may be a tie
    print(f'  -> final refit on {train[H].shape}: {mdl_full.regressor_.best_params_}')
    print(f'     {len(fold_picks)} folds chose:')
    for k, v in chose.items():
        print(f'        {v:2d}x  {k}')
    print(f'     full-set pick is among the modal fold choices: '
          f'{str(mdl_full.regressor_.best_params_) in modal}'
          + ('   (folds tied)' if len(modal) > 1 else ''))

# --- kernel diagnostics over ALL folds; Block 4's length-scale line reads off this ---
# ell/diameter > 1 means the optimiser has stretched the kernel past the whole dataset
# to compensate for assuming more roughness than the data has -- a misspecification
# signature, and the reason ELL_BOUNDS carries a decade of margin (Block 1b).
_kd = pd.DataFrame(kernel_rows)
_kd = _kd[_kd.kernel == NAME]
print()
print(f'  --- {NAME}: fitted kernel over ALL folds ---')
for target in TARGETS:
    _s = _kd[_kd.target == target]
    print(f'  {target:13s} n={len(_s):2d} folds | ell/diameter min {_s.ell_over_diam.min():.2f} '
          f'median {_s.ell_over_diam.median():.2f} max {_s.ell_over_diam.max():.2f} '
          f'| ell > diameter in {int((_s.ell_over_diam > 1).sum())}/{len(_s)} folds '
          f'| LML median {_s.lml.median():.2f}')


=== GPR (Matern 1.5) ===
grid: {'model__alpha': [1e-08, 1e-06, 0.0001, 0.001]}
  permeability fold  0 | held out [1.334, 1.778, 2.371]      | fit on (198, 16) | picked {'model__alpha': 1e-08} | fold MAPE    4.822%
    inner CV: 10 folds, scoring=neg_mean_absolute_error (higher is better)
                     candidate     f0     f1     f2     f3     f4     f5     f6     f7     f8     f9   mean  rank
       {'model__alpha': 1e-08} -0.046 -0.018 -0.013 -0.013 -0.047 -0.071 -0.077 -0.104 -0.091 -0.081 -0.056     1
       {'model__alpha': 1e-06} -0.046 -0.018 -0.013 -0.013 -0.046 -0.071 -0.078 -0.104 -0.091 -0.081 -0.056     2
      {'model__alpha': 0.0001} -0.047 -0.015 -0.010 -0.012 -0.043 -0.080 -0.095 -0.112 -0.082 -0.082 -0.058     3
       {'model__alpha': 0.001} -0.083 -0.018 -0.008 -0.018 -0.051 -0.122 -0.133 -0.116 -0.036 -0.112 -0.070     4
    -> selected {'model__alpha': 1e-08} (mean -0.05611)
    MLE-fitted kernel (the OTHER half of the tuning): ell=26.589  sf=13.027  LML=263.

d:\Study\Softwares\MiniConda\envs\classical\Lib\site-packages\sklearn\gaussian_process\_gpr.py:670: ConvergenceWarning: lbfgs failed to converge after 15 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


In [103]:
NAME = 'GPR (Matern 2.5)'
PIPE = Pipeline([('scaler', StandardScaler()),
                 ('model',  GaussianProcessRegressor(
                     kernel=ConstantKernel(1.0, SF_BOUNDS) * Matern(1.0, ELL_BOUNDS, nu=2.5),
                     normalize_y=True,        # fallback is the training mean, not 0
                     n_restarts_optimizer=0,  # 2 free hyperparameters, unimodal surface:
                                              # 0/1/2/5/10 verified identical
                     random_state=RANDOM_STATE))])
GRID = {'model__alpha': NUGGETS}

print(f'=== {NAME} ===')
print(f'grid: {GRID}')

for target in TARGETS:
    fold_mapes, fold_picks = [], []
    for i, (tr, te, inner) in enumerate(FOLDS[target]):
        mdl = TransformedTargetRegressor(
            GridSearchCV(clone(PIPE), GRID, cv=inner, scoring=SCORING, n_jobs=-1),
            func=TARGET_FUNC[target][0], inverse_func=TARGET_FUNC[target][1])
        mdl.fit(train[H].iloc[tr], train[target].iloc[tr])

        pred = mdl.predict(train[H].iloc[te])
        yt   = train[target].iloc[te].values
        fold_mape = np.mean(np.abs((pred - yt) / yt)) * 100
        fold_mapes.append(fold_mape)
        fold_picks.append(str(mdl.regressor_.best_params_))

        if VERBOSE >= 1:
            shown = '[' + ', '.join(f'{v:.4g}' for v in np.unique(yt)) + ']'
            print(f'  {target:12s} fold {i:2d} | held out {shown:26s} '
                  f'| fit on {train[H].iloc[tr].shape} | picked {mdl.regressor_.best_params_} '
                  f'| fold MAPE {fold_mape:8.3f}%')
        # ---- what the hyperparameter search actually did ----
        gs  = mdl.regressor_
        # Kernel diagnostics for EVERY fold, not just the traced ones, so that the
        # length-scale evidence in Block 4 is read off a table instead of recalled.
        # ell and sf are fitted by marginal likelihood inside .fit(); alpha comes from
        # the inner CV. Recorded together because Block 4 compares them at matched alpha.
        _g = gs.best_estimator_.named_steps['model']
        kernel_rows.append({'kernel': NAME, 'target': target, 'fold': i,
                            'alpha': gs.best_params_['model__alpha'],
                            'ell':   _g.kernel_.k2.length_scale,
                            'sf':    np.sqrt(_g.kernel_.k1.constant_value),
                            'lml':   _g.log_marginal_likelihood_value_,
                            'ell_over_diam': _g.kernel_.k2.length_scale / D_MAX})
        if VERBOSE >= 2 and i < TRACE_FOLDS:
            res = gs.cv_results_
            tune = pd.DataFrame({'candidate': [str(p) for p in res['params']]})
            for j in range(gs.n_splits_):
                tune[f'f{j}'] = res[f'split{j}_test_score']
            tune['mean'] = res['mean_test_score']
            tune['rank'] = res['rank_test_score']
            print(f'    inner CV: {gs.n_splits_} folds, scoring={SCORING} (higher is better)')
            print('      ' + tune.round(5).to_string(index=False).replace('\n', '\n      '))
            print(f'    -> selected {gs.best_params_} (mean {gs.best_score_:.5f})')
            g = _g
            print(f'    MLE-fitted kernel (the OTHER half of the tuning): '
                  f'ell={g.kernel_.k2.length_scale:.3f}  '
                  f'sf={np.sqrt(g.kernel_.k1.constant_value):.3f}  '
                  f'LML={g.log_marginal_likelihood_value_:.2f}  '
                  f'ell/diameter={g.kernel_.k2.length_scale / D_MAX:.2f}')

        rows = train.iloc[te]
        cv_rows.append(pd.DataFrame({'target': target, 'model': NAME,
                                     'permeability': rows.permeability.values,
                                     'resistivity':  rows.resistivity.values,
                                     'y_true': yt, 'y_pred': pred,
                                     'picked': str(mdl.regressor_.best_params_)}))

    print(f'  -> {target}: {len(fold_mapes)} folds, median fold MAPE {np.median(fold_mapes):.3f}%')

    # final refit on ALL 225 TRAINING rows. No test data is involved.
    mdl_full = TransformedTargetRegressor(
        GridSearchCV(clone(PIPE), GRID, cv=FULL_INNER[target], scoring=SCORING, n_jobs=-1),
        func=TARGET_FUNC[target][0], inverse_func=TARGET_FUNC[target][1])
    mdl_full.fit(train[H], train[target])
    FINAL[(target, NAME)] = mdl_full

    # Does the full-set refit agree with what the folds chose? If not, the CV
    # estimate describes a different model from the one that will be tested.
    chose = pd.Series(fold_picks).value_counts()
    modal = set(chose[chose == chose.max()].index)      # may be a tie
    print(f'  -> final refit on {train[H].shape}: {mdl_full.regressor_.best_params_}')
    print(f'     {len(fold_picks)} folds chose:')
    for k, v in chose.items():
        print(f'        {v:2d}x  {k}')
    print(f'     full-set pick is among the modal fold choices: '
          f'{str(mdl_full.regressor_.best_params_) in modal}'
          + ('   (folds tied)' if len(modal) > 1 else ''))

# --- kernel diagnostics over ALL folds; Block 4's length-scale line reads off this ---
# ell/diameter > 1 means the optimiser has stretched the kernel past the whole dataset
# to compensate for assuming more roughness than the data has -- a misspecification
# signature, and the reason ELL_BOUNDS carries a decade of margin (Block 1b).
_kd = pd.DataFrame(kernel_rows)
_kd = _kd[_kd.kernel == NAME]
print()
print(f'  --- {NAME}: fitted kernel over ALL folds ---')
for target in TARGETS:
    _s = _kd[_kd.target == target]
    print(f'  {target:13s} n={len(_s):2d} folds | ell/diameter min {_s.ell_over_diam.min():.2f} '
          f'median {_s.ell_over_diam.median():.2f} max {_s.ell_over_diam.max():.2f} '
          f'| ell > diameter in {int((_s.ell_over_diam > 1).sum())}/{len(_s)} folds '
          f'| LML median {_s.lml.median():.2f}')


=== GPR (Matern 2.5) ===
grid: {'model__alpha': [1e-08, 1e-06, 0.0001, 0.001]}
  permeability fold  0 | held out [1.334, 1.778, 2.371]      | fit on (198, 16) | picked {'model__alpha': 0.0001} | fold MAPE    3.503%
    inner CV: 10 folds, scoring=neg_mean_absolute_error (higher is better)
                     candidate     f0     f1     f2     f3     f4     f5     f6     f7     f8     f9   mean  rank
       {'model__alpha': 1e-08} -0.055 -0.021 -0.021 -0.030 -0.088 -0.050 -0.065 -0.096 -0.124 -0.087 -0.064     3
       {'model__alpha': 1e-06} -0.053 -0.021 -0.021 -0.029 -0.087 -0.050 -0.065 -0.095 -0.124 -0.086 -0.063     2
      {'model__alpha': 0.0001} -0.026 -0.018 -0.012 -0.013 -0.040 -0.039 -0.061 -0.093 -0.134 -0.067 -0.050     1
       {'model__alpha': 0.001} -0.067 -0.020 -0.015 -0.027 -0.055 -0.102 -0.117 -0.102 -0.093 -0.078 -0.068     4
    -> selected {'model__alpha': 0.0001} (mean -0.05017)
    MLE-fitted kernel (the OTHER half of the tuning): ell=6.157  sf=4.719  LML=299.

d:\Study\Softwares\MiniConda\envs\classical\Lib\site-packages\sklearn\gaussian_process\_gpr.py:670: ConvergenceWarning: lbfgs failed to converge after 11 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


  permeability fold  9 | held out [316.2, 421.7]             | fit on (207, 16) | picked {'model__alpha': 0.0001} | fold MAPE   14.912%
  permeability fold 10 | held out [562.3, 749.9]             | fit on (207, 16) | picked {'model__alpha': 0.0001} | fold MAPE    7.143%
  -> permeability: 11 folds, median fold MAPE 4.155%
  -> final refit on (225, 16): {'model__alpha': 0.0001}
     11 folds chose:
         9x  {'model__alpha': 0.0001}
         2x  {'model__alpha': 1e-08}
     full-set pick is among the modal fold choices: True
  resistivity  fold  0 | held out [3e-07]                    | fit on (200, 16) | picked {'model__alpha': 0.0001} | fold MAPE    5.057%
    inner CV: 6 folds, scoring=neg_mean_absolute_error (higher is better)
                     candidate     f0     f1     f2     f3     f4     f5   mean  rank
       {'model__alpha': 1e-08} -0.000 -0.000 -0.000 -0.000 -0.000 -0.000 -0.000     3
       {'model__alpha': 1e-06} -0.000 -0.000 -0.000 -0.000 -0.000 -0.000 -0.000     

### 2e — matched-nugget kernel comparison and convergence census

The two evidence lines Block 4 leans on hardest — marginal likelihood and convergence
— are produced **here**, not recalled. Both need care:

- **Marginal likelihood is only interpretable at matched `alpha`.** The two arms freeze
  at different nuggets, so comparing each at its own pick would confound kernel with
  nugget. This cell fits both kernels at every nugget and pairs them.
- **Convergence cannot be counted from stderr.** Python's default filter shows a warning
  once per (message, category, location), and `GridSearchCV` fits inside joblib workers
  with their own registries — so the number of warnings appearing in a cell's output is
  not a census. These fits are single-process and individually wrapped, so the count is
  exact.

LML is computed on the **transformed** target (log for μᵣ, raw for ρ), so it compares
across kernels *within* a target and never across targets.


In [104]:
# --- 2e: matched-nugget LML comparison + exact convergence census ------------------
# 2 targets x 2 kernels x 4 nuggets = 16 single-process fits on all 225 training rows.
# No test data is involved. Nothing here selects anything -- it measures the two
# quantities Block 4 cites, so that the frozen decision can be audited against a table.
import warnings

KERNEL_NU = {'GPR (Matern 1.5)': 1.5, 'GPR (Matern 2.5)': 2.5}

lml_rows = []
for target in TARGETS:
    for kname, nu in KERNEL_NU.items():
        for a in NUGGETS:
            pipe = Pipeline([('scaler', StandardScaler()),
                             ('model',  GaussianProcessRegressor(
                                 kernel=ConstantKernel(1.0, SF_BOUNDS) * Matern(1.0, ELL_BOUNDS, nu=nu),
                                 alpha=a, normalize_y=True, n_restarts_optimizer=0,
                                 random_state=RANDOM_STATE))])
            mdl = TransformedTargetRegressor(pipe, func=TARGET_FUNC[target][0],
                                             inverse_func=TARGET_FUNC[target][1])
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                mdl.fit(train[H], train[target])
            g = mdl.regressor_.named_steps['model']
            lml_rows.append({
                'target': target, 'kernel': kname, 'alpha': a,
                'alpha_s': f'{a:.0e}',
                'ell': g.kernel_.k2.length_scale,
                'ell_over_diam': g.kernel_.k2.length_scale / D_MAX,
                'lml': g.log_marginal_likelihood_value_,
                'n_conv_warn': sum(c.category.__name__ == 'ConvergenceWarning' for c in caught)})

lml = pd.DataFrame(lml_rows)
print(f'{len(lml)} fits: {lml.target.nunique()} targets x {lml.kernel.nunique()} kernels '
      f'x {lml.alpha.nunique()} nuggets')

# --- paired at matched alpha ---
piv = lml.pivot_table(index=['target', 'alpha_s'], columns='kernel', values='lml').reset_index()
piv['margin_2.5_minus_1.5'] = piv['GPR (Matern 2.5)'] - piv['GPR (Matern 1.5)']
piv['nu=2.5 wins'] = piv['margin_2.5_minus_1.5'] > 0
n_win, n_tot = int(piv['nu=2.5 wins'].sum()), len(piv)
print()
print('log marginal likelihood at MATCHED nugget (higher is better):')
display(piv.rename(columns={'alpha_s': 'alpha'})
           .style.format({'GPR (Matern 1.5)': '{:,.2f}', 'GPR (Matern 2.5)': '{:,.2f}',
                          'margin_2.5_minus_1.5': '{:+,.2f}'}).hide(axis='index'))
print(f'  nu=2.5 wins {n_win}/{n_tot} matched-nugget comparisons | '
      f'margins {piv["margin_2.5_minus_1.5"].min():+.1f} to '
      f'{piv["margin_2.5_minus_1.5"].max():+.1f} nats')

# --- length scale at matched alpha, same 16 fits ---
print()
print('ell/diameter at matched nugget (>1 = kernel stretched past the whole dataset):')
for kname in KERNEL_NU:
    s = lml[lml.kernel == kname]
    print(f'  {kname:18s} min {s.ell_over_diam.min():.2f} max {s.ell_over_diam.max():.2f} '
          f'| ell > diameter in {int((s.ell_over_diam > 1).sum())}/{len(s)} fits')

# --- convergence, counted rather than observed ---
print()
print('convergence census:')
for kname in KERNEL_NU:
    s = lml[lml.kernel == kname]
    print(f'  {kname:18s} {int((s.n_conv_warn > 0).sum())}/{len(s)} fits raised '
          f'ConvergenceWarning')
bad = lml[lml.n_conv_warn > 0]
if len(bad):
    display(bad[['target', 'kernel', 'alpha_s', 'ell_over_diam', 'lml', 'n_conv_warn']]
            .rename(columns={'alpha_s': 'alpha'})
            .style.format({'ell_over_diam': '{:,.2f}', 'lml': '{:,.2f}'}).hide(axis='index'))
else:
    print('  no ConvergenceWarning from any (kernel, target, nugget) combination')


16 fits: 2 targets x 2 kernels x 4 nuggets

log marginal likelihood at MATCHED nugget (higher is better):


target,alpha,GPR (Matern 1.5),GPR (Matern 2.5),margin_2.5_minus_1.5,nu=2.5 wins
permeability,1e-03,328.27,354.71,+26.43,True
permeability,1e-04,362.18,401.15,+38.97,True
permeability,1e-06,362.98,369.19,+6.21,True
permeability,1e-08,362.92,368.70,+5.78,True
resistivity,1e-03,172.60,222.34,+49.74,True
resistivity,1e-04,187.90,226.94,+39.03,True
resistivity,1e-06,189.88,200.74,+10.86,True
resistivity,1e-08,189.90,200.67,+10.77,True


  nu=2.5 wins 8/8 matched-nugget comparisons | margins +5.8 to +49.7 nats

ell/diameter at matched nugget (>1 = kernel stretched past the whole dataset):
  GPR (Matern 1.5)   min 1.54 max 4.63 | ell > diameter in 8/8 fits
  GPR (Matern 2.5)   min 0.26 max 0.68 | ell > diameter in 0/8 fits

convergence census:
  GPR (Matern 1.5)   1/8 fits raised ConvergenceWarning
  GPR (Matern 2.5)   0/8 fits raised ConvergenceWarning


target,kernel,alpha,ell_over_diam,lml,n_conv_warn
resistivity,GPR (Matern 1.5),1e-08,3.76,189.90,1


## Block 3 — CV metrics

All four model blocks wrote into one long frame, so the metric arithmetic is
written **once** here instead of four times. Effective sample size is *levels*
(23 for mu, 7 for rho), not rows — so the median across levels is the summary,
never a pooled mean.

No PASS columns: the 10% / 20% criteria belong to the FE test only.

In [105]:
cv = pd.concat(cv_rows, ignore_index=True)
print(f'cv_rows: {len(cv_rows)} frames -> {cv.shape[0]} rows x {cv.shape[1]} cols')
print(f'  models  {sorted(cv.model.unique())}')
print(f'  targets {sorted(cv.target.unique())}')
assert cv.model.nunique() == 4 and cv.target.nunique() == 2

allp = cv.assign(split='CV')
allp['abs_err']  = (allp.y_pred - allp.y_true).abs()
allp['sq_err']   = (allp.y_pred - allp.y_true) ** 2
allp['abs_pct']  = allp.abs_err / allp.y_true * 100
allp['signed']   = (allp.y_pred - allp.y_true) / allp.y_true * 100

# Bound violations, counted on the RAW predictions -- nothing is clipped first.
_p_lo   = allp.target.map({t: v[0] for t, v in PHYS_BOUNDS.items()})
_p_str  = allp.target.map({t: v[1] for t, v in PHYS_BOUNDS.items()})
allp['unphysical']  = np.where(_p_str, allp.y_pred <= _p_lo, allp.y_pred < _p_lo)
allp['below_range'] = allp.y_pred < allp.target.map({t: v[0] for t, v in RANGE_BOUNDS.items()})
allp['above_range'] = allp.y_pred > allp.target.map({t: v[1] for t, v in RANGE_BOUNDS.items()})

# A raw-target fit can emit a non-positive prediction, for which no log-ratio exists.
# Those rows drop out of `spread` (they are counted as unphysical instead) rather than
# being clipped to a value that would silently dominate the statistic.
allp['logratio'] = np.log(allp.y_pred.where(allp.y_pred > 0) / allp.y_true)

per_level = (allp.groupby(['split', 'target', 'model', 'y_true'], as_index=False)
                 .agg(n=('y_pred', 'size'),
                      mae=('abs_err', 'mean'),
                      rmse=('sq_err', lambda s: np.sqrt(s.mean())),
                      mape=('abs_pct', 'mean'),
                      medape=('abs_pct', 'median'),
                      bias=('signed', 'median'),
                      spread=('logratio', lambda s: s.std(ddof=0) * 100),
                      n_unphys=('unphysical', 'sum'),
                      n_below=('below_range', 'sum'),
                      n_above=('above_range', 'sum')))
print(f'per_level -> {per_level.shape[0]} rows (one per split x target x model x level)')
per_level.head(9)

cv_rows: 72 frames -> 1528 rows x 7 cols
  models  ['GPR (Matern 1.5)', 'GPR (Matern 2.5)', 'Poly2+Ridge', 'kNN (PCA-4)']
  targets ['permeability', 'resistivity']
per_level -> 120 rows (one per split x target x model x level)


,split,target,model,y_true,n,mae,rmse,mape,medape,bias,spread,n_unphys,n_below,n_above
0,CV,permeability,GPR (Matern 1.5),1.334,9,0.085,0.086,6.349,6.068,-6.068,1.052,0,0,0
1,CV,permeability,GPR (Matern 1.5),1.778,9,0.107,0.112,6.007,6.494,-6.494,1.952,0,0,0
2,CV,permeability,GPR (Matern 1.5),2.371,9,0.050,0.061,2.109,2.489,-2.489,1.533,0,0,0
3,CV,permeability,GPR (Matern 1.5),3.162,9,0.026,0.033,0.835,0.432,-0.432,0.865,0,0,0
4,CV,permeability,GPR (Matern 1.5),4.217,9,0.041,0.047,0.984,1.129,1.129,0.565,0,0,0
5,CV,permeability,GPR (Matern 1.5),5.623,9,0.110,0.114,1.955,1.901,-1.901,0.540,0,0,0
6,CV,permeability,GPR (Matern 1.5),7.499,9,0.040,0.043,0.528,0.417,-0.417,0.359,0,0,0
7,CV,permeability,GPR (Matern 1.5),10.000,9,0.110,0.138,1.095,1.083,-1.083,0.961,0,0,0
8,CV,permeability,GPR (Matern 1.5),13.335,9,0.174,0.204,1.305,1.222,-1.213,1.267,0,0,0


In [106]:
# Compare each model against the floor, level by level (paired, not ratio-of-medians).
floor_tbl = (per_level[per_level.model == FLOOR][['split', 'target', 'y_true', 'mape']]
             .rename(columns={'mape': 'floor_mape'}))
pl = per_level.merge(floor_tbl, on=['split', 'target', 'y_true'], how='left')
pl['ratio'] = pl.floor_mape / pl.mape

summary = (pl.groupby(['split', 'target', 'model'], as_index=False)
             .agg(n_levels=('mape', 'size'),
                  mape_med=('mape', 'median'),
                  mape_max=('mape', 'max'),
                  medape_med=('medape', 'median'),
                  bias_med=('bias', 'median'),
                  spread_med=('spread', 'median'),
                  n_unphys=('n_unphys', 'sum'),
                  n_below=('n_below', 'sum'),
                  n_above=('n_above', 'sum'),
                  vs_floor=('ratio', 'median'),
                  wins=('ratio', lambda s: (s > 1).sum())))
summary['wins'] = summary.wins.astype(str) + '/' + summary.n_levels.astype(str)

for t in TARGETS:
    print(f'===== CV: {t} (model selection -- no PASS columns) =====')
    display(summary.query('split == "CV" and target == @t').drop(columns=['split', 'target']))

===== CV: permeability (model selection -- no PASS columns) =====


,model,n_levels,mape_med,mape_max,medape_med,bias_med,spread_med,n_unphys,n_below,n_above,vs_floor,wins
0,GPR (Matern 1.5),23,6.007,11.935,6.068,-0.432,3.891,0,0,0,5.069,23/23
1,GPR (Matern 2.5),23,4.232,15.930,3.096,0.763,4.131,0,0,0,5.662,23/23
2,Poly2+Ridge,23,9.235,15.375,5.445,0.634,7.861,0,0,0,3.093,22/23
3,kNN (PCA-4),23,32.164,77.759,32.193,-4.014,12.515,0,0,1,1.000,0/23


===== CV: resistivity (model selection -- no PASS columns) =====


,model,n_levels,mape_med,mape_max,medape_med,bias_med,spread_med,n_unphys,n_below,n_above,vs_floor,wins
4,GPR (Matern 1.5),7,0.783,6.666,0.522,0.266,0.984,0,0,0,13.339,7/7
5,GPR (Matern 2.5),7,0.406,5.057,0.342,-0.107,0.540,0,0,0,25.669,7/7
6,Poly2+Ridge,7,4.712,15.714,4.065,-0.557,5.312,0,0,0,3.360,7/7
7,kNN (PCA-4),7,11.000,40.167,7.500,0.000,13.226,0,0,0,1.000,0/7


## Block 4 — freeze the decision

Written **before** the FE cell runs, so the choice can be dated. The justification
below uses only evidence that does not involve the FE set.

In [107]:
# === DECISION, frozen before Block 5 runs ===
# ERRATA -- two rounds, both against re-measurements of evidence that never touches the
# FE set, and the selection is unchanged in both: nu=2.5 on every leg that survives.
# Correcting misquoted figures from a run already in hand is an erratum; re-selecting on
# new evidence would have unfrozen this block.
#   (1) earlier: three figures quoted here were wrong against a re-measurement of the
#       same CV run.
#   (2) 2026-08-27: the LML and LENGTH SCALE lines were quoted from a measurement that
#       NO CELL IN THIS NOTEBOOK PRODUCED. Both quantities only ever appeared inside the
#       `VERBOSE >= 2 and i < TRACE_FOLDS` trace -- 4 of 18 folds -- and nothing
#       aggregated them. Block 2e and the per-fold `kernel_rows` collection were added so
#       that each line now reads off a printed table. Figures below are updated, and each
#       is labelled with the measurement that produces it.
#
# GPR nu=2.5 over nu=1.5, on evidence that never touches the FE data:
#
#   - MARGINAL LIKELIHOOD favours 2.5 on BOTH targets, at EVERY nugget in the grid --
#     8/8 matched-nugget comparisons, margins +5.8 to +49.7 nats (Block 2e). Comparing at
#     matched alpha matters: the two arms freeze at different nuggets (1e-6/1e-8 vs
#     1e-4/1e-4), so a comparison at their own picks alone would confound kernel with
#     nugget.  [was "6 to 39 nats" -- wrong at both ends.]
#
#   - LENGTH SCALE: nu=1.5 stretches ell PAST the whole dataset diameter; nu=2.5 never
#     does. Measured two ways, both now printed:
#       * matched nugget, full training set (Block 2e): nu=1.5 ell/diameter 1.54-4.63,
#         above 1 in 8/8 fits; nu=2.5 0.26-0.68, in 0/8.
#       * per outer fold: the all-folds summary at the end of Blocks 2c/2d, built from
#         `kernel_rows`, prints min/median/max and "ell > diameter in n/n folds" for each
#         target. The fold ranges previously quoted here came from the unaggregated
#         trace and are deliberately NOT restated -- read them off that table.
#     nu=1.5 is stretching the kernel past the data to compensate for assuming more
#     roughness than the data has.
#
#   - CONVERGENCE, scoped to what was actually counted: at matched nugget on the FULL
#     TRAINING SET (Block 2e), nu=1.5 fails to converge in 1/8 fits and nu=2.5 in 0/8,
#     and that one failure is exactly nu=1.5, rho, alpha=1e-8 (ell/diameter 3.76). It is
#     a flat likelihood at that stretched length scale, and n_restarts_optimizer does not
#     move it -- the same misspecification the length-scale line describes, showing up in
#     the optimiser.
#     [was "the only ConvergenceWarning this notebook emits ... nu=2.5 converges at every
#     nugget on both targets". False as written: Block 2d (nu=2.5) also emits one, from a
#     fold fit inside GridSearchCV. And a stderr count is not a census in either
#     direction -- the default warning filter fires once per (message, category,
#     location), and n_jobs=-1 fits run in joblib workers with their own registries, so
#     repeats are dropped silently. Block 2e wraps every fit in catch_warnings(record=
#     True), which is why its counts are exact. PER-FOLD convergence remains uncensused.]
#
#   - NUGGET STABILITY ACROSS FOLDS supports the choice on mu ONLY, and is kept here
#     rather than dropped because on rho it points the other way. nu=2.5: stable on mu
#     (9/11 pick 1e-4), NOT stable on rho (4x 1e-3 / 3x 1e-4, and the full-set refit
#     takes the minority 1e-4). nu=1.5 is the reverse -- three ways on mu (4/4/3), 6/7
#     on rho. The rho ambiguity is measured in 7c and is worth 0.017 pts of MAPE.
SELECTED = 'GPR (Matern 2.5)'

print(f'SELECTED = {SELECTED}')
for target in TARGETS:
    print(f'  {target:13s} final hyperparameters: '
          f'{FINAL[(target, SELECTED)].regressor_.best_params_}')

SELECTED = GPR (Matern 2.5)
  permeability  final hyperparameters: {'model__alpha': 0.0001}
  resistivity   final hyperparameters: {'model__alpha': 0.0001}


## Block 5 — sealed FE test

**The only cell in this notebook that predicts `test` features.** Everything above
uses `train` only. Run once; anything changed in response to these numbers is
post-hoc and has to be labelled as such.

All four models are reported, not just the selected one — they were all frozen in
Blocks 2a–2d, so reporting them is not selection.

In [108]:
fe_rows = []
for (target, name), mdl in FINAL.items():
    pred = mdl.predict(test[H])          # <-- the only use of test[H]
    print(f'  {name:18s} {target:13s} -> {len(pred)} predictions | '
          f'range [{pred.min():.4g}, {pred.max():.4g}] | true [{test[target].min():.4g}, '
          f'{test[target].max():.4g}]')
    fe_rows.append(pd.DataFrame({'target': target, 'model': name,
                                 'permeability': test.permeability.values,
                                 'resistivity':  test.resistivity.values,
                                 'y_true': test[target].values, 'y_pred': pred,
                                 'picked': str(mdl.regressor_.best_params_)}))

fe = pd.concat(fe_rows, ignore_index=True)
print(f'\nfe: {fe.shape[0]} rows = {fe.model.nunique()} models x '
      f'{fe.target.nunique()} targets x {len(test)} FE rows')

  kNN (PCA-4)        permeability  -> 55 predictions | range [31.93, 1000] | true [50, 1000]
  kNN (PCA-4)        resistivity   -> 55 predictions | range [2.375e-07, 9.875e-07] | true [2e-07, 1e-06]
  Poly2+Ridge        permeability  -> 55 predictions | range [6.523e-15, 7.821e+09] | true [50, 1000]
  Poly2+Ridge        resistivity   -> 55 predictions | range [-0.0001078, 2.897e-05] | true [2e-07, 1e-06]
  GPR (Matern 1.5)   permeability  -> 55 predictions | range [20.55, 2193] | true [50, 1000]
  GPR (Matern 1.5)   resistivity   -> 55 predictions | range [-8.375e-08, 1.206e-06] | true [2e-07, 1e-06]
  GPR (Matern 2.5)   permeability  -> 55 predictions | range [20.88, 2323] | true [50, 1000]
  GPR (Matern 2.5)   resistivity   -> 55 predictions | range [-1.037e-07, 1.037e-06] | true [2e-07, 1e-06]

fe: 440 rows = 4 models x 2 targets x 55 FE rows


## Block 6 — FE results

Same metric arithmetic as Block 3, recomputed over CV **and** FE together so the two
are directly comparable.

**Four metrics on the overall target**, because MAPE alone hides two different failures:
`mae` and `rmse` in the target's own units (dimensionless for μᵣ, ohm m for ρ),
`mape` as the stated criterion, and `medape` — the median absolute percentage error —
to show whether the mean is being carried by a minority of rows. `rmse` ≫ `mae` means a
few large misses; `mape` ≫ `medape` means the same thing in relative terms.

`mape` is the criterion as written (a mean over all rows). `mape_med_level` is the
median across levels — the same scale as the per-level table, and robust to a single
level running away.

**Bounds are counted, never enforced.** The next cell reports every prediction that is
unphysical (μᵣ < 1, ρ ≤ 0) or outside the training range, on the raw predictions.
No clipping is applied anywhere in this notebook, so these counts are diagnostic rather
than cosmetic.

In [109]:
allp = pd.concat([cv.assign(split='CV'), fe.assign(split='FE')], ignore_index=True)
allp['abs_err']  = (allp.y_pred - allp.y_true).abs()
allp['sq_err']   = (allp.y_pred - allp.y_true) ** 2
allp['abs_pct']  = allp.abs_err / allp.y_true * 100
allp['signed']   = (allp.y_pred - allp.y_true) / allp.y_true * 100

# Bound violations, counted on the RAW predictions -- nothing is clipped first.
_p_lo   = allp.target.map({t: v[0] for t, v in PHYS_BOUNDS.items()})
_p_str  = allp.target.map({t: v[1] for t, v in PHYS_BOUNDS.items()})
allp['unphysical']  = np.where(_p_str, allp.y_pred <= _p_lo, allp.y_pred < _p_lo)
allp['below_range'] = allp.y_pred < allp.target.map({t: v[0] for t, v in RANGE_BOUNDS.items()})
allp['above_range'] = allp.y_pred > allp.target.map({t: v[1] for t, v in RANGE_BOUNDS.items()})

# A raw-target fit can emit a non-positive prediction, for which no log-ratio exists.
# Those rows drop out of `spread` (they are counted as unphysical instead) rather than
# being clipped to a value that would silently dominate the statistic.
allp['logratio'] = np.log(allp.y_pred.where(allp.y_pred > 0) / allp.y_true)

per_level = (allp.groupby(['split', 'target', 'model', 'y_true'], as_index=False)
                 .agg(n=('y_pred', 'size'),
                      mae=('abs_err', 'mean'),
                      rmse=('sq_err', lambda s: np.sqrt(s.mean())),
                      mape=('abs_pct', 'mean'),
                      medape=('abs_pct', 'median'),
                      bias=('signed', 'median'),
                      spread=('logratio', lambda s: s.std(ddof=0) * 100),
                      n_unphys=('unphysical', 'sum'),
                      n_below=('below_range', 'sum'),
                      n_above=('above_range', 'sum')))

# Four metrics on the overall target, so the picture does not rest on MAPE alone.
# MAE and RMSE carry the target's own units (dimensionless for mu_r, ohm m for rho);
# MAPE is the stated criterion; median APE shows whether the mean is being driven by a
# minority of rows. RMSE >> MAE is the signature of a few large misses.
overall = (allp.groupby(['split', 'target', 'model'], as_index=False)
               .agg(n=('y_pred', 'size'),
                    mae=('abs_err', 'mean'),
                    rmse=('sq_err', lambda s: np.sqrt(s.mean())),
                    mape=('abs_pct', 'mean'),
                    medape=('abs_pct', 'median'),
                    n_unphys=('unphysical', 'sum'),
                    n_below=('below_range', 'sum'),
                    n_above=('above_range', 'sum')))

fe_tbl = (per_level.query('split == "FE"')
          .groupby(['target', 'model'], as_index=False)
          .agg(n_levels=('mape', 'size'),
               mape_med_level=('mape', 'median'),
               mape_worst_level=('mape', 'max'))
          .merge(overall.query('split == "FE"').drop(columns='split'),
                 on=['target', 'model']))
fe_tbl['PASS_overall']   = fe_tbl.mape <= PASS_OVERALL
fe_tbl['PASS_per_level'] = fe_tbl.mape_worst_level <= PASS_LEVEL
fe_tbl = fe_tbl[['target', 'model', 'n', 'mae', 'rmse', 'mape', 'medape',
                 'mape_med_level', 'mape_worst_level', 'n_levels',
                 'n_unphys', 'n_below', 'n_above', 'PASS_overall', 'PASS_per_level']]

for t in TARGETS:
    print(f'===== FE TEST: {t} -- four metrics on the overall target =====')
    display(fe_tbl.query('target == @t').drop(columns='target')
                  .style.format({'mae': '{:,.4g}', 'rmse': '{:,.4g}',
                                 'mape': '{:,.2f}', 'medape': '{:,.2f}',
                                 'mape_med_level': '{:,.2f}',
                                 'mape_worst_level': '{:,.2f}'}).hide(axis='index'))

===== FE TEST: permeability -- four metrics on the overall target =====


model,n,mae,rmse,mape,medape,mape_med_level,mape_worst_level,n_levels,n_unphys,n_below,n_above,PASS_overall,PASS_per_level
GPR (Matern 1.5),55,466.7,639.8,89.33,58.73,84.55,137.14,11,0,0,21,False,False
GPR (Matern 2.5),55,502.7,706,94.20,59.08,89.57,144.79,11,0,0,21,False,False
Poly2+Ridge,55,1.655e+08,1.059e+09,"286,986,759.56",100.00,"624,421.83","3,128,407,596.75",11,9,9,25,False,False
kNN (PCA-4),55,185.4,279.8,36.61,36.14,38.50,47.26,11,0,0,2,False,False


===== FE TEST: resistivity -- four metrics on the overall target =====


model,n,mae,rmse,mape,medape,mape_med_level,mape_worst_level,n_levels,n_unphys,n_below,n_above,PASS_overall,PASS_per_level
GPR (Matern 1.5),55,3.112e-07,3.606e-07,58.12,54.08,49.25,99.55,5,4,11,3,False,False
GPR (Matern 2.5),55,3.191e-07,3.762e-07,59.45,55.87,50.91,104.26,5,4,10,1,False,False
Poly2+Ridge,55,1.699e-05,2.678e-05,"5,432.80","1,805.02","1,615.96","21,161.31",5,31,32,23,False,False
kNN (PCA-4),55,2.255e-07,3.001e-07,69.82,21.88,21.36,238.64,5,0,0,0,False,False


In [110]:
# --- bounds check, on the raw predictions, BEFORE any clipping (none is applied) ---
# Two separate questions, deliberately not merged:
#   unphysical   -- the value cannot exist for this material class (mu_r < 1, rho <= 0)
#   below/above  -- the value is outside the training support, so the model is
#                   extrapolating; physically possible, but unsupported by the data
for t in TARGETS:
    lo_p, strict = PHYS_BOUNDS[t]
    lo_r, hi_r   = RANGE_BOUNDS[t]
    print(f'=== {t} | physical: {"x >" if strict else "x >="} {lo_p:g}'
          f' | training range: [{lo_r:g}, {hi_r:g}] ===')
    for split in ['CV', 'FE']:
        sub = allp.query('split == @split and target == @t')
        for model in sorted(sub.model.unique()):
            m = sub[sub.model == model]
            flag = '  <-- UNPHYSICAL' if m.unphysical.any() else ''
            print(f'  {split} {model:18s} n={len(m):4d} | unphysical {int(m.unphysical.sum()):3d} '
                  f'| below range {int(m.below_range.sum()):3d} | above range {int(m.above_range.sum()):3d} '
                  f'| pred [{m.y_pred.min():.4g}, {m.y_pred.max():.4g}]{flag}')
    print()

viol = allp[allp.unphysical]
if len(viol):
    print('Every unphysical prediction, listed in full (nothing is clipped):')
    display(viol[['split', 'target', 'model', 'permeability', 'resistivity',
                  'y_true', 'y_pred']]
            .sort_values(['target', 'model', 'y_pred'])
            .style.format({'permeability': '{:,.4g}', 'resistivity': '{:,.3g}',
                           'y_true': '{:,.4g}', 'y_pred': '{:,.4g}'}).hide(axis='index'))
else:
    print('No unphysical predictions from any model on either split.')

=== permeability | physical: x >= 1 | training range: [1, 1000] ===
  CV GPR (Matern 1.5)   n= 207 | unphysical   0 | below range   0 | above range   0 | pred [1.228, 881.8]
  CV GPR (Matern 2.5)   n= 207 | unphysical   0 | below range   0 | above range   0 | pred [1.254, 850.9]
  CV Poly2+Ridge        n= 207 | unphysical   0 | below range   0 | above range   0 | pred [1.264, 896.9]
  CV kNN (PCA-4)        n= 207 | unphysical   0 | below range   0 | above range   1 | pred [2.14, 1000]
  FE GPR (Matern 1.5)   n=  55 | unphysical   0 | below range   0 | above range  21 | pred [20.55, 2193]
  FE GPR (Matern 2.5)   n=  55 | unphysical   0 | below range   0 | above range  21 | pred [20.88, 2323]
  FE Poly2+Ridge        n=  55 | unphysical   9 | below range   9 | above range  25 | pred [6.523e-15, 7.821e+09]  <-- UNPHYSICAL
  FE kNN (PCA-4)        n=  55 | unphysical   0 | below range   0 | above range   2 | pred [31.93, 1000]

=== resistivity | physical: x > 0 | training range: [2e-07, 1e-0

split,target,model,permeability,resistivity,y_true,y_pred
FE,permeability,Poly2+Ridge,50,8e-07,50,6.523e-15
FE,permeability,Poly2+Ridge,50,1e-06,50,3.199e-13
FE,permeability,Poly2+Ridge,50,6e-07,50,3.132e-11
FE,permeability,Poly2+Ridge,100,1e-06,100,1.405e-10
FE,permeability,Poly2+Ridge,100,8e-07,100,3.342e-08
FE,permeability,Poly2+Ridge,200,1e-06,200,0.002336
FE,permeability,Poly2+Ridge,100,6e-07,100,0.005686
FE,permeability,Poly2+Ridge,200,8e-07,200,0.3883
FE,permeability,Poly2+Ridge,300,1e-06,300,0.9974
FE,resistivity,GPR (Matern 1.5),500,2e-07,2e-07,-8.375e-08


In [111]:
print(f'--- per-level, {SELECTED} | MAPE and median APE ---')
for t in TARGETS:
    sub = per_level.query('split == "FE" and target == @t and model == @SELECTED').copy()
    sub['PASS'] = sub.mape <= PASS_LEVEL
    if t == 'resistivity':
        sub['y_true'] = sub.y_true.map('{:.0e}'.format)
    display(sub[['y_true', 'n', 'mae', 'rmse', 'mape', 'medape', 'bias',
                 'n_unphys', 'n_below', 'n_above', 'PASS']]
            .rename(columns={'y_true': 'level'})
            .style.format({'mae': '{:,.4g}', 'rmse': '{:,.4g}', 'mape': '{:,.2f}',
                           'medape': '{:,.2f}', 'bias': '{:,.2f}'}).hide(axis='index'))

--- per-level, GPR (Matern 2.5) | MAPE and median APE ---


level,n,mae,rmse,mape,medape,bias,n_unphys,n_below,n_above,PASS
50.000000,5,16.22,19.22,32.44,23.21,-23.21,0,0,0,False
100.000000,5,41.61,45.77,41.61,43.94,-9.59,0,0,0,False
200.000000,5,169,209.9,84.52,60.43,59.08,0,0,0,False
300.000000,5,347.4,444,115.81,89.79,89.79,0,0,1,False
400.000000,5,562.5,695.6,140.63,117.96,117.96,0,0,2,False
500.000000,5,723.9,878.1,144.79,128.30,128.30,0,0,3,False
600.000000,5,769.8,955.8,128.31,121.59,121.59,0,0,3,False
700.000000,5,753.4,950.2,107.62,102.50,102.50,0,0,3,False
800.000000,5,716.6,902.9,89.57,77.85,77.85,0,0,3,False
900.000000,5,720.1,845.9,80.01,56.48,52.43,0,0,3,False


level,n,mae,rmse,mape,medape,bias,n_unphys,n_below,n_above,PASS
2e-07,11,2.085e-07,2.318e-07,104.26,125.11,-45.70,4,7,0,False
4e-07,11,1.755e-07,2.044e-07,43.87,48.41,-29.78,0,3,0,False
6e-07,11,2.664e-07,2.895e-07,44.39,47.22,-45.04,0,0,0,False
8e-07,11,4.073e-07,4.362e-07,50.91,53.79,-53.79,0,0,0,False
1e-06,11,5.38e-07,5.813e-07,53.80,60.06,-60.06,0,0,1,False


In [112]:
# Decade coverage. FE has NO levels below mu=10 and only one in [10,100), so a
# per-decade table here can support a top-decade claim and nothing else.
DECADE_LABELS = {0: '[1,10)', 1: '[10,100)', 2: '[100,1000]'}
fe_mu = np.sort(test.permeability.unique())
fe_dec = np.clip(np.floor(np.log10(fe_mu)).astype(int), 0, 2)

print('FE decade coverage:')
for d in (0, 1, 2):
    print(f'  {DECADE_LABELS[d]:12s} {int((fe_dec == d).sum()):2d} levels  '
          f'{fe_mu[fe_dec == d].tolist()}')

sub = per_level.query('split == "FE" and target == "permeability" and model == @SELECTED').copy()
sub['decade'] = np.clip(np.floor(np.log10(sub.y_true)).astype(int), 0, 2).map(DECADE_LABELS)
dec_tbl = (sub.groupby('decade', as_index=False)
              .agg(n_levels=('mape', 'size'), n_rows=('n', 'sum'),
                   mape_med=('mape', 'median'), mape_min=('mape', 'min'),
                   mape_max=('mape', 'max'),
                   medape_med=('medape', 'median'),
                   medape_min=('medape', 'min'), medape_max=('medape', 'max'),
                   n_unphys=('n_unphys', 'sum'), n_below=('n_below', 'sum'),
                   n_above=('n_above', 'sum')))
# med/min/max collapse to one number where a decade holds a single level.
dec_tbl['single_level'] = dec_tbl.n_levels == 1
display(dec_tbl.style.format({'mape_med': '{:,.2f}', 'mape_min': '{:,.2f}', 'mape_max': '{:,.2f}',
                              'medape_med': '{:,.2f}', 'medape_min': '{:,.2f}',
                              'medape_max': '{:,.2f}'}).hide(axis='index'))

FE decade coverage:
  [1,10)        0 levels  []
  [10,100)      1 levels  [50]
  [100,1000]   10 levels  [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]


decade,n_levels,n_rows,mape_med,mape_min,mape_max,medape_med,medape_min,medape_max,n_unphys,n_below,n_above,single_level
"[10,100)",1,5,32.44,32.44,32.44,23.21,23.21,23.21,0,0,0,True
"[100,1000]",10,50,98.60,41.61,144.79,83.82,43.94,128.30,0,0,21,False


## Block 7 — post-hoc transfer check

**Post-hoc. This block reads `test`, and nothing in it may feed back into model
selection** — Block 4 was frozen before Block 5 ran and stays frozen.

The question: at the material combinations shared by the dense and FE grids, compare
predictions made from *dense-model* spectra against predictions made from *FE*
spectra.

- only the FE spectra fail → model-to-model transfer, or feature alignment
- both fail → the held-out split, or the inverse model

Two things that framing leaves open, which this block supplies:

1. **A threshold.** "Fail" needs a reference. Each held-out level is compared against
   its *own* CV per-level MAPE from Block 3, read out of `per_level` rather than
   hardcoded — the closest thing to a like-for-like number this notebook contains.
2. **Transfer vs alignment.** The paired test alone cannot separate them: both
   produce the same symptom of FE failing while dense passes. Cell 7b separates them
   by magnitude, without fitting anything.

**Two limits to carry into any write-up.** The shared permeability levels are 100 and
1000, both inside `[100, 1000]`, so this block speaks to the top decade only. And the
protocol anchors (μᵣ=1000, ρ=2e-7, ρ=1e-6) are never held out, so their rows are
in-sample and are flagged `held_out=False` — illustrative, not evidence.

In [113]:
# --- 7a: the shared material combinations, both spectra side by side ---
# test.permeability is int64 and train.permeability is float64; cast before merging.
PAIR = train.merge(test.astype({'permeability': float}),
                   on=['permeability', 'resistivity'], suffixes=('_d', '_f'))

# The suffixed merge yields H_imag_375_d / H_imag_375_f. Renaming back to H is what
# guarantees the model receives its 16 features in the order it learned them -- the
# single most likely way this comparison could silently answer the wrong question.
Xd = PAIR[[c + '_d' for c in H]].set_axis(H, axis=1)   # dense-model spectra
Xf = PAIR[[c + '_f' for c in H]].set_axis(H, axis=1)   # FE spectra

assert len(PAIR) == 10, f'expected 10 shared combinations, got {len(PAIR)}'
assert list(Xd.columns) == list(Xf.columns) == H, 'feature order does not match H'
assert not PAIR.duplicated(['permeability', 'resistivity']).any(), 'duplicated combination'

FMT = {'permeability': '{:g}', 'resistivity': '{:.0e}'}
print(f'{len(PAIR)} shared material combinations')
for t in TARGETS:
    anchors = set(np.sort(train[t].unique())[[0, -1]])
    shared  = sorted(PAIR[t].unique())
    blocked = sorted(anchors & set(shared))
    show    = lambda vs: ', '.join(FMT[t].format(v) for v in vs)
    print(f'  {t:13s} {len(shared)} shared levels [{show(shared)}]')
    print(f'  {"":13s} {len(blocked)} of them anchors, cannot be held out: [{show(blocked)}]')

10 shared material combinations
  permeability  2 shared levels [100, 1000]
                1 of them anchors, cannot be held out: [1000]
  resistivity   5 shared levels [2e-07, 4e-07, 6e-07, 8e-07, 1e-06]
                2 of them anchors, cannot be held out: [2e-07, 1e-06]


In [114]:
# --- 7b: model-free displacement, and what a real misalignment would look like ---
# Standardise on TRAINING data only, so both frames live in one space, one unit per
# training SD. At these ten points the material is IDENTICAL, so a correctly aligned
# pair should land on nearly the same point; whatever distance remains is the
# disagreement between the two forward models, measured in the units the model uses.
sc = StandardScaler().fit(train[H])
Zd = sc.transform(Xd)

FREQ = sorted({int(c.rsplit('_', 1)[1]) for c in H})
LEX  = sorted(FREQ, key=str)        # the order a text-typed frequency column would give

# Deliberately wrong column orders, to calibrate the scale an alignment bug occupies.
ORDERS = {
    'correctly aligned (actual)':     H,
    'frequencies ordered as text':    [f'H_imag_{f}' for f in LEX] + [f'H_real_{f}' for f in LEX],
    'frequency order reversed':       [f'H_imag_{f}' for f in FREQ[::-1]] + [f'H_real_{f}' for f in FREQ[::-1]],
    'H_real / H_imag blocks swapped': [f'H_real_{f}' for f in FREQ] + [f'H_imag_{f}' for f in FREQ],
}

disp_rows = []
for name, cols in ORDERS.items():
    d = np.linalg.norm(sc.transform(Xf[cols].set_axis(H, axis=1)) - Zd, axis=1)
    disp_rows.append({'feature order': name, 'median SD': np.median(d), 'max SD': d.max()})
    if name.startswith('correctly'):
        disp_actual = d
disp_tbl = pd.DataFrame(disp_rows)

# Reference scales, both already established earlier in this notebook.
Ztr = sc.transform(train[H])
d2  = ((Ztr[:, None, :] - Ztr[None, :, :]) ** 2).sum(-1)
np.fill_diagonal(d2, np.inf)
nn_train = np.sqrt(d2.min(1))

print(f'dense grid, nearest-neighbour spacing : median {np.median(nn_train):.3f} SD')
print(f'dataset diameter (D_MAX, Block 1b)    : {D_MAX:.2f} SD')
print(f'FE vs dense at identical material     : median {np.median(disp_actual):.3f} SD, '
      f'max {disp_actual.max():.3f} SD')
print(f'  -> {np.median(disp_actual) / np.median(nn_train):.1f}x the grid spacing, '
      f'{np.median(disp_actual) / D_MAX:.3f}x the diameter\n')
display(disp_tbl.style.format({'median SD': '{:,.3f}', 'max SD': '{:,.3f}'}).hide(axis='index'))

print('\nA scrambled feature order displaces the FE rows by two orders of magnitude more')
print('than what is observed, because standardisation divides each channel by its OWN')
print('training SD: feeding H_real_48000 (~3.5e-4) into the slot the model expects')
print('H_imag_48000 (~3e-6) in produces hundreds of SD, not a few. Alignment is ruled')
print('out by magnitude alone -- what remains is a forward-model difference.')

dense grid, nearest-neighbour spacing : median 0.268 SD
dataset diameter (D_MAX, Block 1b)    : 13.37 SD
FE vs dense at identical material     : median 0.505 SD, max 2.099 SD
  -> 1.9x the grid spacing, 0.038x the diameter



feature order,median SD,max SD
correctly aligned (actual),0.505,2.099
frequencies ordered as text,101.042,102.709
frequency order reversed,151.083,154.937
H_real / H_imag blocks swapped,188.989,192.164



A scrambled feature order displaces the FE rows by two orders of magnitude more
than what is observed, because standardisation divides each channel by its OWN
training SD: feeding H_real_48000 (~3.5e-4) into the slot the model expects
H_imag_48000 (~3e-6) in produces hundreds of SD, not a few. Alignment is ruled
out by magnitude alone -- what remains is a forward-model difference.


In [115]:
# --- 7c: paired comparison, with the level held out so the dense side is honest ---
# mu=100 and mu=1000 are TRAINING levels. Predicting them with the Block 4 final model
# would be in-sample and would report ~0.5% for reasons that have nothing to do with
# transfer. So: drop the level, refit, then predict BOTH spectra at that level.
#
# The pipeline is written out literally rather than reused from Block 2d -- `PIPE`
# there is a loop variable, and depending on it is exactly the leak this notebook
# avoids elsewhere. Hyperparameters come from the frozen final models; the inner
# GridSearchCV is not re-run because the nugget makes almost no difference here. On mu
# it is stable anyway (9/11 folds pick 1e-4). On rho it is NOT stable -- the folds split
# 4x 1e-3 / 3x 1e-4 and the refit takes the minority 1e-4 -- but refitting the rho folds
# at each fixed nugget puts the two candidates 0.017 pts apart (median fold MAPE 0.423%
# at 1e-4 against 0.406% at 1e-3, over the same 7 folds). Re-running the search would
# not move this comparison, and would cost readability.
assert SELECTED == 'GPR (Matern 2.5)', \
    f'7c writes out the Matern 2.5 kernel literally; SELECTED is {SELECTED}'

pair_rows = []
for target in TARGETS:
    best = FINAL[(target, SELECTED)].regressor_.best_params_
    assert set(best) == {'model__alpha'}, f'unexpected hyperparameters: {best}'
    anchors = set(np.sort(train[target].unique())[[0, -1]])

    for lvl in sorted(PAIR[target].unique()):
        # An anchor cannot be held out without turning the prediction into
        # extrapolation, which is a different question. Keep it, and flag it.
        held = lvl not in anchors
        keep = ~np.isclose(train[target].values, lvl) if held else np.ones(len(train), bool)

        pipe = Pipeline([('scaler', StandardScaler()),
                         ('model',  GaussianProcessRegressor(
                             kernel=ConstantKernel(1.0, SF_BOUNDS) * Matern(1.0, ELL_BOUNDS, nu=2.5),
                             normalize_y=True, n_restarts_optimizer=0,
                             random_state=RANDOM_STATE))]).set_params(**best)
        mdl = TransformedTargetRegressor(pipe, func=TARGET_FUNC[target][0], inverse_func=TARGET_FUNC[target][1])
        mdl.fit(train[H][keep], train[target][keep])

        sel = np.isclose(PAIR[target].values, lvl)
        y   = PAIR[target].values[sel]
        p_d = mdl.predict(Xd[sel])      # dense-model spectra
        p_f = mdl.predict(Xf[sel])      # FE spectra, same materials, same model

        mape_d = np.mean(np.abs(p_d - y) / y) * 100
        mape_f = np.mean(np.abs(p_f - y) / y) * 100
        pair_rows.append({'target': target, 'level': lvl, 'n': int(sel.sum()),
                          'held_out': held, 'n_train': int(keep.sum()),
                          'mape_dense': mape_d, 'mape_fe': mape_f})
        print(f'  {target:13s} level {lvl:<9.4g} held_out={str(held):5s} '
              f'fit on {int(keep.sum()):3d} rows | dense {mape_d:8.2f}% | FE {mape_f:8.2f}%')

pair_tbl = pd.DataFrame(pair_rows)
print(f'\npair_tbl: {len(pair_tbl)} rows, {int(pair_tbl.held_out.sum())} of them held out')

  permeability  level 100       held_out=True  fit on 216 rows | dense     3.56% | FE    47.68%
  permeability  level 1000      held_out=False fit on 225 rows | dense     0.21% | FE    70.90%
  resistivity   level 2e-07     held_out=False fit on 225 rows | dense     0.43% | FE    74.03%
  resistivity   level 4e-07     held_out=True  fit on 200 rows | dense     3.70% | FE    32.78%
  resistivity   level 6e-07     held_out=True  fit on 200 rows | dense     0.40% | FE    12.31%
  resistivity   level 8e-07     held_out=True  fit on 200 rows | dense     0.12% | FE    23.98%
  resistivity   level 1e-06     held_out=False fit on 225 rows | dense     0.01% | FE    33.24%

pair_tbl: 7 rows, 4 of them held out


In [116]:
# --- 7d: verdict ---
# The bar for each level is its OWN CV per-level MAPE from Block 3. Anchors have no CV
# row (they are never held out), so they merge to NaN -- which is the correct signal
# that those rows carry no evidential weight.
cv_ref = (per_level.query('split == "CV" and model == @SELECTED')
                   [['target', 'y_true', 'mape']]
                   .rename(columns={'y_true': 'level', 'mape': 'mape_cv'}))
tbl = pair_tbl.merge(cv_ref, on=['target', 'level'], how='left')
tbl['dense_vs_cv'] = tbl.mape_dense / tbl.mape_cv
tbl['fe_vs_cv']    = tbl.mape_fe / tbl.mape_cv

show = tbl.copy()
show['level'] = show.level.map(lambda v: f'{v:.4g}')
display(show[['target', 'level', 'n', 'held_out', 'n_train', 'mape_cv',
              'mape_dense', 'mape_fe', 'dense_vs_cv', 'fe_vs_cv']]
        .style.format({'mape_cv': '{:,.2f}', 'mape_dense': '{:,.2f}', 'mape_fe': '{:,.2f}',
                       'dense_vs_cv': '{:,.1f}x', 'fe_vs_cv': '{:,.1f}x'}, na_rep='--')
        .hide(axis='index'))

honest = tbl[tbl.held_out]
print('\n--- held-out levels only ---')
for t in TARGETS:
    h = honest[honest.target == t]
    print(f'{t:13s} n={len(h)} | dense {h.mape_dense.min():6.2f}-{h.mape_dense.max():6.2f}% '
          f'({h.dense_vs_cv.max():.1f}x CV at worst) | '
          f'FE {h.mape_fe.min():6.2f}-{h.mape_fe.max():6.2f}% '
          f'({h.fe_vs_cv.max():.1f}x CV at worst)')

# Thresholds stated rather than implied: within 3x its own CV value is "in line with
# cross-validation"; beyond 5x is a different regime.
dense_ok = (honest.dense_vs_cv <= 3).all()
fe_bad   = (honest.fe_vs_cv > 5).any()
print()
if dense_ok and fe_bad:
    print('Only the FE spectra fail. The held-out split and the inverse model are')
    print('exonerated at these materials: the same model, at the same held-out levels,')
    print('reproduces its CV accuracy on dense spectra and loses one to two orders of')
    print('magnitude on FE spectra. Cause is model-to-model transfer or feature')
    print('alignment -- and 7b rules out alignment by magnitude, leaving transfer.')
elif not dense_ok:
    print('Both spectra fail -> the held-out split or the inverse model is implicated;')
    print('the FE comparison cannot be read as a transfer result until that is resolved.')
else:
    print('Neither side fails at these materials; the FE failure lies outside the')
    print('top decade this block can see.')

target,level,n,held_out,n_train,mape_cv,mape_dense,mape_fe,dense_vs_cv,fe_vs_cv
permeability,100,5,True,216,6.31,3.56,47.68,0.6x,7.6x
permeability,1000,5,False,225,--,0.21,70.90,--,--
resistivity,2e-07,2,False,225,--,0.43,74.03,--,--
resistivity,4e-07,2,True,200,1.91,3.70,32.78,1.9x,17.2x
resistivity,6e-07,2,True,200,0.41,0.40,12.31,1.0x,30.3x
resistivity,8e-07,2,True,200,0.17,0.12,23.98,0.7x,141.5x
resistivity,1e-06,2,False,225,--,0.01,33.24,--,--



--- held-out levels only ---
permeability  n=1 | dense   3.56-  3.56% (0.6x CV at worst) | FE  47.68- 47.68% (7.6x CV at worst)
resistivity   n=3 | dense   0.12-  3.70% (1.9x CV at worst) | FE  12.31- 32.78% (141.5x CV at worst)

Only the FE spectra fail. The held-out split and the inverse model are
exonerated at these materials: the same model, at the same held-out levels,
reproduces its CV accuracy on dense spectra and loses one to two orders of
magnitude on FE spectra. Cause is model-to-model transfer or feature
alignment -- and 7b rules out alignment by magnitude, leaving transfer.


## Block 8 — export for the robustness studies

Freezes the four refitted estimators, the train-fitted scaler and the feature spec to
`models/`. `notebooks/robustness/01_liftoff.ipynb` and `02_noise.ipynb` load these and
apply them to perturbed spectra without refitting — the export is what lets "no
retraining" be checked rather than asserted.

In [117]:
# --- Block 8: export the fitted estimators -------------------------------------
# The robustness studies (notebooks/robustness/) apply these models to liftoff-shifted
# and noise-perturbed spectra, and must not refit anything. Persisting FINAL here is
# what makes that claim auditable: those notebooks load estimators, never call .fit().
import joblib

MODEL_DIR = Path.cwd().parents[1] / 'models'
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(FINAL, MODEL_DIR / 'final_models.joblib')

# Train-fitted scaler, for the SD-displacement diagnostic ONLY. Every estimator in
# FINAL is a TransformedTargetRegressor(GridSearchCV(Pipeline([... StandardScaler ...])))
# and so scales internally -- predictions are taken on the RAW 16-column frame, exactly
# as Block 5 does. Pre-scaling with this would double-scale and silently ruin the study.
joblib.dump(StandardScaler().fit(train[H]), MODEL_DIR / 'train_scaler.joblib')

# The canonical feature order, so downstream code never re-derives it. reshape_wide
# emits the H_imag block first (pivot_table alphabetises values=), which is not the
# order src/data.py reads top to bottom. Block 7b measured the cost of getting this
# wrong: 101-189 SD of displacement against 0.505 SD when correct.
joblib.dump({'H': H,
             'freqs': sorted({int(c.rsplit('_', 1)[1]) for c in H}),
             'targets': list(TARGETS),
             'phys_bounds': PHYS_BOUNDS,
             'range_bounds': RANGE_BOUNDS,
             'selected': SELECTED},
            MODEL_DIR / 'feature_spec.joblib')

print(f'wrote {len(FINAL)} estimators -> {MODEL_DIR}')
for f in sorted(MODEL_DIR.iterdir()):
    print(f'  {f.name:24s} {f.stat().st_size / 1024:8.1f} KB')

wrote 8 estimators -> d:\Study\Uni\ect-inversion\models
  feature_spec.joblib           0.5 KB
  final_models.joblib        1997.9 KB
  train_scaler.joblib           1.4 KB
